In [9]:
# Importar las bibliotecas necesarias
from langchain import OpenAI, LLMChain
from langchain.prompts import PromptTemplate
import pandas as pd

# Configurar la clave de API de OpenAI
import openai
openai.api_key = "TU_API_KEY"

# Crear una base de datos ficticia de lecciones y ejercicios
data = {
    "curso": ["Deep Learning Básico", "Machine Learning Avanzado", "Python para IA"],
    "tema": ["Redes Neuronales", "Optimización de Modelos", "Limpieza de Datos"],
    "lección": ["Introducción a Redes Neuronales", "Regularización y Dropout", "Trabajando con Pandas"],
    "ejercicio": ["Ejercicio 1: Construcción de una red neuronal", 
                  "Ejercicio 2: Implementación de dropout en TensorFlow", 
                  "Ejercicio 3: Limpieza de datos con Pandas"],
    "nivel_dificultad": ["Básico", "Avanzado", "Intermedio"]
}

df = pd.DataFrame(data)

# Mostrar la base de datos
print("Base de datos ficticia:")
print(df)

# Configurar prompts y agentes

## Agente 1: Procesador de consultas
prompt_agente1 = PromptTemplate(
    input_variables=["consulta"],
    template="""
    Eres un agente de procesamiento de consultas. Clasifica la consulta del usuario en una de las siguientes categorías:
    - Lecciones
    - Ejercicios
    - Otros
    La consulta es: {consulta}
    Responde con la categoría en una sola palabra.
    """
)

agente1 = LLMChain(
    llm=OpenAI(temperature=0.7, model="gpt-4"),
    prompt=prompt_agente1
)

## Agente 2: Buscador en la base de datos
def buscar_en_base_datos(categoria, consulta):
    if categoria.lower() == "lecciones":
        resultado = df[df["tema"].str.contains(consulta, case=False)]
        if not resultado.empty:
            return resultado[["lección", "curso"]].to_dict(orient="records")
        else:
            return "No se encontraron resultados para la consulta."
    elif categoria.lower() == "ejercicios":
        resultado = df[df["tema"].str.contains(consulta, case=False)]
        if not resultado.empty:
            return resultado[["ejercicio", "curso"]].to_dict(orient="records")
        else:
            return "No se encontraron resultados para la consulta."
    else:
        return "Consulta fuera de las categorías admitidas."

## Agente 3: Generador de respuestas
prompt_agente3 = PromptTemplate(
    input_variables=["resultados"],
    template="""
    Eres un agente generador de respuestas. Con base en los siguientes resultados: {resultados},
    genera una respuesta detallada para el usuario indicando la lección o ejercicio más relevante y su descripción.
    """
)

agente3 = LLMChain(
    llm=OpenAI(temperature=0.7, model="gpt-4"),
    prompt=prompt_agente3
)

# Flujo de trabajo

def flujo_de_trabajo(consulta_usuario):
    # Paso 1: Procesar la consulta
    categoria = agente1.run({"consulta": consulta_usuario})
    print(f"Categoría detectada: {categoria}")

    # Paso 2: Buscar en la base de datos
    resultados = buscar_en_base_datos(categoria, consulta_usuario)
    print(f"Resultados de la búsqueda: {resultados}")

    # Paso 3: Generar respuesta
    if isinstance(resultados, list):  # Si se encontraron resultados
        respuesta = agente3.run({"resultados": resultados})
        print("Respuesta generada por el sistema:")
        print(respuesta)
    else:
        print(resultados)

# Ejemplo práctico: Ejecutar el flujo de trabajo
consulta_ejemplo = "Redes Neuronales"
flujo_de_trabajo(consulta_ejemplo)


Base de datos ficticia:
                       curso                     tema  \
0       Deep Learning Básico         Redes Neuronales   
1  Machine Learning Avanzado  Optimización de Modelos   
2             Python para IA        Limpieza de Datos   

                           lección  \
0  Introducción a Redes Neuronales   
1         Regularización y Dropout   
2            Trabajando con Pandas   

                                           ejercicio nivel_dificultad  
0      Ejercicio 1: Construcción de una red neuronal           Básico  
1  Ejercicio 2: Implementación de dropout en Tens...         Avanzado  
2          Ejercicio 3: Limpieza de datos con Pandas       Intermedio  


ValidationError: 1 validation error for OpenAI
  Value error, Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. [type=value_error, input_value={'temperature': 0.7, 'mod...ne, 'http_client': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error